# Train vs Dev Postposition Comparison

This notebook applies the **same methodology** as `02_postposition_analysis.ipynb` to the **development split** and compares results with the training split.

**Goal:** Determine whether postposition evidence used in `docs/rule_specification_v1.md` generalizes beyond the training set.

**Focus postpositions:** `ने`, `को`, `से`, `में`, `पर`

No rule specification changes are made here — analysis only.

## Methodology (same as notebook 02)

For every token with `deprel = case`:
1. Record the postposition (`form`)
2. Record the `deprel` of its head (parent)
3. Count frequencies per postposition

**Percent formula:**

```
percent = 100 × count(parent_deprel = X for postposition P) / total(case form = P)
```

## 1. Load Data and Extract Case Records

In [1]:
from collections import Counter, defaultdict
from pathlib import Path


def load_conllu(filepath):
    """Read a CONLL-U file and return a list of sentence dictionaries."""
    sentences = []
    current = {"tokens": []}

    with open(filepath, encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")

            if not line:
                if current["tokens"]:
                    sentences.append(current)
                    current = {"tokens": []}
                continue

            if line.startswith("#"):
                continue

            columns = line.split("\t")
            if len(columns) < 8 or "-" in columns[0]:
                continue

            current["tokens"].append({
                "id": columns[0],
                "form": columns[1],
                "head": columns[6],
                "deprel": columns[7],
            })

    if current["tokens"]:
        sentences.append(current)

    return sentences


def extract_case_records(sentences):
    """Return (postposition, parent_deprel) tuples for all case tokens."""
    records = []

    for sentence in sentences:
        id_to_token = {token["id"]: token for token in sentence["tokens"]}

        for token in sentence["tokens"]:
            if token["deprel"] != "case":
                continue

            parent = id_to_token.get(token["head"])
            if parent is None:
                continue

            records.append((token["form"], parent["deprel"]))

    return records


def build_postposition_index(case_records):
    """Group parent deprels by postposition form."""
    index = defaultdict(list)
    for postposition, parent_deprel in case_records:
        index[postposition].append(parent_deprel)
    return index


TRAIN_PATH = Path("../data/raw/hi_hdtb-ud-train.conllu")
DEV_PATH = Path("../data/raw/hi_hdtb-ud-dev.conllu")

train_records = extract_case_records(load_conllu(TRAIN_PATH))
dev_records = extract_case_records(load_conllu(DEV_PATH))

train_by_pp = build_postposition_index(train_records)
dev_by_pp = build_postposition_index(dev_records)

print(f"Train: {len(train_records)} case tokens")
print(f"Dev:   {len(dev_records)} case tokens")

Train: 53121 case tokens
Dev:   6674 case tokens


## 2. Comparison Helper

In [2]:
FOCUS_POSTPOSITIONS = ["ने", "को", "से", "में", "पर"]

# Parent labels cited in rule_specification_v1.md per postposition
RULE_RELEVANT_LABELS = {
    "ने": ["nsubj", "conj", "nsubj:pass", "obl", "obj"],
    "को": ["obj", "iobj", "obl", "nsubj"],
    "से": ["obl", "nmod", "obj", "iobj"],
    "में": ["obl", "nmod", "conj"],
    "पर": ["obl", "nmod", "conj"],
}


def percent(count, total):
    """Return percentage, or 0 if total is zero."""
    return 100 * count / total if total else 0.0


def compare_parent_label(postposition, parent_label, train_index, dev_index):
    """Return train %, dev %, absolute difference, and counts."""
    train_total = len(train_index[postposition])
    dev_total = len(dev_index[postposition])

    train_count = Counter(train_index[postposition])[parent_label]
    dev_count = Counter(dev_index[postposition])[parent_label]

    train_pct = percent(train_count, train_total)
    dev_pct = percent(dev_count, dev_total)
    abs_diff = abs(train_pct - dev_pct)

    return {
        "postposition": postposition,
        "parent_label": parent_label,
        "train_count": train_count,
        "train_total": train_total,
        "train_pct": train_pct,
        "dev_count": dev_count,
        "dev_total": dev_total,
        "dev_pct": dev_pct,
        "abs_diff": abs_diff,
    }


def print_comparison_table(rows, title):
    """Print a train vs dev comparison table."""
    print(title)
    print(
        f"{'Parent label':<14} {'Train %':>8} {'Dev %':>8} {'|Diff|':>8}  "
        f"{'Train n':>8} {'Dev n':>8}"
    )
    print("-" * 62)

    for row in rows:
        print(
            f"{row['parent_label']:<14} "
            f"{row['train_pct']:>7.1f}% {row['dev_pct']:>7.1f}% {row['abs_diff']:>7.1f}pp  "
            f"{row['train_count']:>8} {row['dev_count']:>8}"
        )
    print()

## 3. Per-Postposition Comparisons

Tables show **Train %**, **Dev %**, and **absolute difference** (percentage points) for parent labels referenced in the v1 rule specification.

In [3]:
all_comparisons = []

for postposition in FOCUS_POSTPOSITIONS:
    rows = [
        compare_parent_label(postposition, label, train_by_pp, dev_by_pp)
        for label in RULE_RELEVANT_LABELS[postposition]
    ]
    all_comparisons.extend(rows)

    train_total = len(train_by_pp[postposition])
    dev_total = len(dev_by_pp[postposition])
    title = (
        f"Postposition: {postposition}  "
        f"(train n={train_total}, dev n={dev_total})"
    )
    print_comparison_table(rows, title)

Postposition: ने  (train n=4862, dev n=569)
Parent label    Train %    Dev %   |Diff|   Train n    Dev n
--------------------------------------------------------------
nsubj             98.4%    97.7%     0.7pp      4785      556
conj               1.3%     2.1%     0.8pp        62       12
nsubj:pass         0.2%     0.2%     0.0pp        10        1
obl                0.1%     0.0%     0.1pp         3        0
obj                0.0%     0.0%     0.0pp         1        0

Postposition: को  (train n=5799, dev n=675)
Parent label    Train %    Dev %   |Diff|   Train n    Dev n
--------------------------------------------------------------
obj               45.2%    47.7%     2.5pp      2624      322
iobj              24.0%    26.2%     2.3pp      1389      177
obl               17.6%    13.3%     4.3pp      1021       90
nsubj              9.7%     9.2%     0.5pp       560       62

Postposition: से  (train n=4324, dev n=542)
Parent label    Train %    Dev %   |Diff|   Train n    Dev n

## 4. R4 Derived Statistic: `से` + `obj`/`iobj` Combined

The rule specification cites a combined ~15.7% for `से` on `obj`/`iobj` parents. This is derived (not printed in notebook 02).

In [4]:
pp = "से"
train_total = len(train_by_pp[pp])
dev_total = len(dev_by_pp[pp])

train_c = Counter(train_by_pp[pp])
dev_c = Counter(dev_by_pp[pp])

train_combined = percent(train_c["obj"] + train_c["iobj"], train_total)
dev_combined = percent(dev_c["obj"] + dev_c["iobj"], dev_total)
abs_diff = abs(train_combined - dev_combined)

print("Derived: से + (obj OR iobj) parent")
print(f"{'Metric':<28} {'Train %':>8} {'Dev %':>8} {'|Diff|':>8}")
print("-" * 56)
print(
    f"{'obj + iobj combined':<28} "
    f"{train_combined:>7.1f}% {dev_combined:>7.1f}% {abs_diff:>7.1f}pp"
)

Derived: से + (obj OR iobj) parent
Metric                        Train %    Dev %   |Diff|
--------------------------------------------------------
obj + iobj combined             15.7%    15.1%     0.6pp


## 5. Summary: Rule-Critical Statistics

One-row-per-rule view of the parent-label percentages that support R1–R5.

In [5]:
RULE_CRITICAL = [
    ("R1", "ने", "nsubj"),
    ("R2", "में", "obl"),
    ("R3", "पर", "obl"),
    ("R4", "से", "obl"),
    ("R5", "को", "obj"),
    ("R5", "को", "iobj"),
    ("R5", "को", "obl"),
    ("R5", "को", "nsubj"),
]

print(f"{'Rule':<6} {'PP':<6} {'Parent':<10} {'Train %':>8} {'Dev %':>8} {'|Diff|':>8}")
print("-" * 52)

for rule_id, pp, parent_label in RULE_CRITICAL:
    row = compare_parent_label(pp, parent_label, train_by_pp, dev_by_pp)
    print(
        f"{rule_id:<6} {pp:<6} {parent_label:<10} "
        f"{row['train_pct']:>7.1f}% {row['dev_pct']:>7.1f}% {row['abs_diff']:>7.1f}pp"
    )

Rule   PP     Parent      Train %    Dev %   |Diff|
----------------------------------------------------
R1     ने     nsubj         98.4%    97.7%     0.7pp
R2     में    obl           87.6%    85.9%     1.7pp
R3     पर     obl           89.8%    88.7%     1.1pp
R4     से     obl           69.5%    70.8%     1.3pp
R5     को     obj           45.2%    47.7%     2.5pp
R5     को     iobj          24.0%    26.2%     2.3pp
R5     को     obl           17.6%    13.3%     4.3pp
R5     को     nsubj          9.7%     9.2%     0.5pp


## 6. Generalization Assessment (Evidence Only)

Interpretation guidelines for rule evidence — **not** a rule change.

| Rule | Train % | Dev % | |Diff| | Generalization note |
|------|--------:|------:|-------:|---------------------|
| R1 (`ने`+`nsubj`) | 98.4% | 97.7% | 0.7pp | Strong — pattern holds on dev |
| R2 (`में`+`obl`) | 87.6% | 85.9% | 1.7pp | Strong — slight drop, still dominant |
| R3 (`पर`+`obl`) | 89.8% | 88.7% | 1.1pp | Strong — pattern holds |
| R4 (`से`+`obl`) | 69.5% | 70.8% | 1.3pp | Stable — ambiguity unchanged |
| R5 (`को`+`obj`) | 45.2% | 47.7% | 2.5pp | Mixed — largest shift among R5 labels |
| R5 (`को`+`iobj`) | 24.0% | 26.2% | 2.3pp | Mixed — still second most common |
| R5 (`को`+`obl`) | 17.6% | 13.3% | 4.3pp | Watch — dev drop; `को` remains mixed |

**Overall:**
- **R1, R2, R3** evidence generalizes well (all critical parent-label differences ≤ 1.7pp, except R2 at 1.7pp).
- **R4** remains ambiguous on both splits; `obl` dominance is stable (~70%).
- **R5** remains mixed on both splits; `को`+`obl` shows the largest dev shift (4.3pp) but `obj`/`iobj` still dominate.
- No rule-critical statistic reverses rank order between train and dev (top parent label unchanged for all five postpositions).

Record detailed notes in `docs/research_notes.md` if needed.